# Módulo 06: Programación orientada a objetos

Autor: Juliho Castillo Colmenares

> Lección complementaria del ejercicio de este módulo: [enunciado](m06-programacion-orientada-a-objetos-guia.ipynb) y [solución](m06-programacion-orientada-a-objetos-solucion.ipynb). No es una tarea entregada del programa.

Con funciones separamos la lógica en piezas. Pero a menudo los datos y las operaciones sobre ellos van juntos: una cuenta bancaria tiene un saldo y se puede depositar o retirar; un sensor tiene un identificador y unas lecturas que se pueden promediar. La **programación orientada a objetos** agrupa ambos en una sola unidad. En ciencia de datos se usa a diario: un `DataFrame` de pandas o un modelo de scikit-learn son objetos con atributos y métodos.

1. Clase y objeto
2. `__init__` y `self`
3. Atributos de instancia y de clase
4. Métodos
5. Diccionarios como tablas de búsqueda
6. `__repr__` y `__str__`
7. Herencia y `super()`
8. Encapsulación por convención
9. Errores frecuentes
10. Ejercicio

## 1. Clase y objeto

Una **clase** es un molde: describe qué datos guarda y qué operaciones admite. Un **objeto** (o **instancia**) es un ejemplar concreto creado a partir de ese molde. De una misma clase se pueden crear tantos objetos como se quiera, cada uno con sus propios datos.

| Concepto | Analogía | Ejemplo en Python |
| --- | --- | --- |
| Clase | Plano de una casa | `class Cuenta:` |
| Objeto | Una casa construida | `mi_cuenta = Cuenta(...)` |
| Atributo | Dato del objeto | `mi_cuenta.saldo` |
| Método | Acción del objeto | `mi_cuenta.depositar(100)` |

In [1]:
class Punto:
    pass                 # clase vacía, solo para ver la idea

p1 = Punto()
p2 = Punto()
print(type(p1))
print(p1 is p2)          # son dos objetos distintos

<class '__main__.Punto'>
False


## 2. `__init__` y `self`

`__init__` es el método que se ejecuta automáticamente al crear un objeto. Sirve para darle sus datos iniciales. Su primer parámetro, `self`, es **el propio objeto que se está creando**; a través de él se guardan los atributos (`self.nombre = ...`).

Al llamar `Cuenta("Ana", 500)`, Python crea el objeto y ejecuta `__init__(objeto, "Ana", 500)`: `self` se pasa solo, sin escribirlo.

In [2]:
class Cuenta:
    def __init__(self, titular, saldo=0):
        self.titular = titular      # atributo de instancia
        self.saldo = saldo

cuenta_ana = Cuenta("Ana", 500)
cuenta_luis = Cuenta("Luis")

print(cuenta_ana.titular, cuenta_ana.saldo)
print(cuenta_luis.titular, cuenta_luis.saldo)

Ana 500
Luis 0


## 3. Atributos de instancia y de clase

- Un **atributo de instancia** se define con `self.` dentro de un método y es propio de cada objeto.
- Un **atributo de clase** se define directamente en el cuerpo de la clase y lo **comparten** todos los objetos. Es útil para constantes o valores comunes a la clase.

In [3]:
class Producto:
    iva = 0.16                         # atributo de clase: igual para todos

    def __init__(self, nombre, precio):
        self.nombre = nombre           # atributos de instancia
        self.precio = precio

cafe = Producto("café", 80)
pan = Producto("pan", 25)

print(cafe.iva, pan.iva)               # ambos leen el mismo valor
Producto.iva = 0.08                    # cambiarlo en la clase afecta a todos
print(cafe.iva, pan.iva)

0.16 0.16
0.08 0.08


## 4. Métodos

Un **método** es una función definida dentro de la clase. Recibe `self` como primer parámetro, así que puede leer y modificar los atributos del objeto. Se llama con la notación de punto: `objeto.metodo(argumentos)`.

In [4]:
class Cuenta:
    def __init__(self, titular, saldo=0):
        self.titular = titular
        self.saldo = saldo

    def depositar(self, monto):
        self.saldo += monto
        return self.saldo

    def retirar(self, monto):
        if monto > self.saldo:
            return "Fondos insuficientes"
        self.saldo -= monto
        return self.saldo

cuenta = Cuenta("Ana", 500)
print(cuenta.depositar(200))
print(cuenta.retirar(1000))
print(cuenta.retirar(300))

700
Fondos insuficientes
400


## 5. Diccionarios como tablas de búsqueda

Muchos métodos necesitan traducir un símbolo a un valor: un código de país a su nombre, una letra a una calificación, un mes a su número de días. En lugar de una larga cadena de `if`/`elif`, se guarda la correspondencia en un **diccionario** como atributo y se consulta con `[]` o `get`.

La tabla puede ser atributo de clase (si es igual para todos los objetos) o de instancia (si se crea en `__init__`).

In [5]:
class Calendario:
    dias_por_mes = {
        "ene": 31, "feb": 28, "mar": 31, "abr": 30, "may": 31, "jun": 30,
        "jul": 31, "ago": 31, "sep": 30, "oct": 31, "nov": 30, "dic": 31,
    }

    def dias(self, mes):
        return self.dias_por_mes.get(mes.lower(), "Mes desconocido")

    def dias_acumulados(self, meses):
        total = 0
        for m in meses:
            total += self.dias_por_mes[m]
        return total

cal = Calendario()
print(cal.dias("Abr"))
print(cal.dias("xyz"))
print(cal.dias_acumulados(["ene", "feb", "mar"]))   # primer trimestre

30
Mes desconocido
90


## 6. `__repr__` y `__str__`

Al imprimir un objeto sin más, Python muestra algo como `<__main__.Cuenta object at 0x...>`, que no dice nada útil. Dos métodos especiales lo resuelven:

| Método | Lo usa | Pensado para |
| --- | --- | --- |
| `__repr__` | `repr(obj)`, la consola, listas de objetos | Programadores: idealmente, código que recrea el objeto |
| `__str__` | `print(obj)`, `str(obj)`, f-strings | Usuarios: texto legible |

Si solo se define `__repr__`, `print` también lo usa.

In [6]:
class Temperatura:
    def __init__(self, grados):
        self.grados = grados

    def __repr__(self):
        return f"Temperatura({self.grados})"

    def __str__(self):
        return f"{self.grados} °C"

t = Temperatura(21.5)
print(t)                 # usa __str__
print(repr(t))           # usa __repr__
print([t, Temperatura(30)])   # dentro de una lista se usa __repr__

21.5 °C
Temperatura(21.5)
[Temperatura(21.5), Temperatura(30)]


## 7. Herencia y `super()`

Una clase puede **heredar** de otra: la clase hija recibe todos los atributos y métodos de la clase padre y puede añadir otros nuevos o **sobrescribir** los existentes. `super()` permite llamar al método de la clase padre, lo que evita repetir su código, por ejemplo en `__init__`.

In [7]:
class CuentaAhorro(Cuenta):
    def __init__(self, titular, saldo=0, tasa=0.05):
        super().__init__(titular, saldo)   # reutiliza el __init__ de Cuenta
        self.tasa = tasa                   # atributo nuevo

    def aplicar_interes(self):             # método nuevo
        self.saldo = round(self.saldo * (1 + self.tasa), 2)
        return self.saldo

ahorro = CuentaAhorro("Marta", 1000, tasa=0.04)
print(ahorro.depositar(500))       # heredado de Cuenta
print(ahorro.aplicar_interes())    # propio de CuentaAhorro
print(isinstance(ahorro, Cuenta))  # una CuentaAhorro también es una Cuenta

1500
1560.0
True


## 8. Encapsulación por convención

Python no tiene atributos realmente privados. Por **convención**, un nombre que empieza con guion bajo (`_saldo`) indica "uso interno: no se modifica desde fuera". El acceso sigue siendo posible, pero quien lo hace sabe que se salta las reglas de la clase. Los cambios deberían pasar por métodos que validen los datos.

In [8]:
class Termostato:
    def __init__(self, objetivo=21):
        self._objetivo = objetivo          # uso interno

    def fijar(self, grados):
        if not 10 <= grados <= 30:
            return "Fuera de rango"
        self._objetivo = grados
        return self._objetivo

    def objetivo(self):
        return self._objetivo

termo = Termostato()
print(termo.fijar(45))      # el método rechaza el valor
print(termo.fijar(23))
print(termo.objetivo())

Fuera de rango
23
23


## 9. Errores frecuentes

**Olvidar `self` en la definición del método.** Python pasa el objeto como primer argumento de forma automática; si el método no tiene un parámetro para recibirlo, la llamada falla con un `TypeError`.

In [9]:
class Saludo:
    def hola():                    # falta self
        return "hola"

try:
    Saludo().hola()
except TypeError as error:
    print("TypeError:", error)

class Saludo:
    def hola(self):
        return "hola"

print(Saludo().hola())

TypeError: Saludo.hola() takes 0 positional arguments but 1 was given
hola


**Olvidar `self.` al guardar un atributo.** `nombre = nombre` dentro de `__init__` crea una variable local que desaparece al terminar el método; el objeto no guarda nada.

In [10]:
class Mascota:
    def __init__(self, nombre):
        nombre = nombre            # variable local, no atributo

try:
    print(Mascota("Rex").nombre)
except AttributeError as error:
    print("AttributeError:", error)

AttributeError: 'Mascota' object has no attribute 'nombre'


**Usar una lista como atributo de clase cuando debería ser de instancia.** Un atributo de clase mutable lo comparten todos los objetos: añadir un elemento desde uno lo añade "a todos".

In [11]:
class Carrito:
    productos = []                 # compartido por todos los carritos

    def agregar(self, p):
        self.productos.append(p)

c1, c2 = Carrito(), Carrito()
c1.agregar("leche")
print("c2 sin agregar nada:", c2.productos)

class Carrito:
    def __init__(self):
        self.productos = []        # una lista nueva por objeto

    def agregar(self, p):
        self.productos.append(p)

c1, c2 = Carrito(), Carrito()
c1.agregar("leche")
print("c2 con atributo de instancia:", c2.productos)

c2 sin agregar nada: ['leche']
c2 con atributo de instancia: []


**Buscar en el diccionario una clave que no existe.** Con `tabla[clave]`, un símbolo inesperado (por ejemplo, en minúsculas cuando la tabla está en mayúsculas) lanza `KeyError`. Conviene normalizar la entrada antes de buscar, y usar `get` o validar con `in` cuando la clave puede faltar.

## 10. Ejercicio

Se quiere modelar las calificaciones de un estudiante con letras, donde cada letra vale una cantidad de puntos: A = 4, B = 3, C = 2, D = 1 y F = 0.

1. Crea una clase `Expediente` con `__init__(self, nombre)` que guarde el nombre y una lista vacía de calificaciones como atributos de instancia. La tabla de puntos debe ser un diccionario como atributo de clase.
2. Añade un método `agregar(letra)` que acepte la letra en mayúsculas o minúsculas y devuelva `"Letra inválida"` si no está en la tabla.
3. Añade un método `promedio()` que devuelva el promedio de puntos redondeado a 2 decimales (0 si no hay calificaciones).
4. Define `__repr__` para que muestre, por ejemplo, `Expediente('Ana', 3 calificaciones)`.
5. Crea una subclase `ExpedienteBeca` que reciba además un `minimo` (por defecto 3.5), use `super()` en su `__init__` y tenga un método `conserva_beca()` que indique si el promedio alcanza el mínimo.

### Solución

In [12]:
class Expediente:
    puntos = {"A": 4, "B": 3, "C": 2, "D": 1, "F": 0}

    def __init__(self, nombre):
        self.nombre = nombre
        self.calificaciones = []

    def agregar(self, letra):
        letra = letra.upper()
        if letra not in self.puntos:
            return "Letra inválida"
        self.calificaciones.append(letra)

    def promedio(self):
        if not self.calificaciones:
            return 0
        total = sum(self.puntos[c] for c in self.calificaciones)
        return round(total / len(self.calificaciones), 2)

    def __repr__(self):
        return f"Expediente({self.nombre!r}, {len(self.calificaciones)} calificaciones)"


class ExpedienteBeca(Expediente):
    def __init__(self, nombre, minimo=3.5):
        super().__init__(nombre)
        self.minimo = minimo

    def conserva_beca(self):
        return self.promedio() >= self.minimo


ana = Expediente("Ana")
for letra in ["A", "b", "A"]:
    ana.agregar(letra)
print(ana, "->", ana.promedio())
print(ana.agregar("Z"))

luis = ExpedienteBeca("Luis")
for letra in ["A", "B", "C", "A"]:
    luis.agregar(letra)
print(luis, "->", luis.promedio(), "| conserva la beca:", luis.conserva_beca())

Expediente('Ana', 3 calificaciones) -> 3.67
Letra inválida
Expediente('Luis', 4 calificaciones) -> 3.25 | conserva la beca: False


## Resumen

- Una **clase** es un molde y un **objeto** es un ejemplar suyo con sus propios datos; `__init__` los inicializa y `self` es el propio objeto.
- Los atributos de instancia son de cada objeto; los de clase son compartidos, por lo que no deben ser listas u otros mutables que cada objeto necesite por separado.
- Los **métodos** son funciones de la clase que operan sobre `self`; un diccionario como atributo sirve de tabla de búsqueda en lugar de muchos `elif`.
- `__repr__` y `__str__` dan una representación legible del objeto.
- La **herencia** reutiliza una clase existente y `super()` llama a sus métodos; el prefijo `_` marca los atributos de uso interno.

En el **Módulo 07** se presenta pandas, cuyas `Series` y `DataFrame` son objetos con decenas de métodos para analizar datos.